In [1]:
import os
import torch
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.models import vgg16
import torch.nn as nn
from PIL import Image
import random

# Đường dẫn đến mô hình đã lưu
model_path = '/content/drive/MyDrive/KHKT/modelmodel7classvgg16.h5'

# Đường dẫn đến thư mục chứa dữ liệu
data_path = '/content/drive/MyDrive/KHKT/Dataset(7class)'

# Định nghĩa các biến biến đổi dữ liệu hình ảnh
transformations = transforms.Compose([transforms.ToTensor(), transforms.Resize((224, 224))])

# Tạo dataset từ dữ liệu hình ảnh cho thư mục test
test_dataset = ImageFolder(root=os.path.join(data_path, 'test'), transform=transformations)

# Tạo DataLoader cho tập kiểm tra
test_loader = DataLoader(test_dataset, batch_size=32)

# Tạo mô hình VGG16 với trọng số đã được đào tạo trước đó
model = vgg16(pretrained=False)  # Không sử dụng trọng số được đào tạo trước
model.classifier[6] = nn.Linear(4096, 7)  # Chỉnh sửa lớp cuối cùng để phù hợp với số lớp đầu ra

# Load trọng số của mô hình đã lưu
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
model.eval()

# Chọn thiết bị tính toán (GPU hoặc CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Định nghĩa danh sách tên lớp
class_names = ["cardboard", "e-waste", "glass", "medical", "metal", "paper", "plastic"]

# Hàm để hiển thị dự đoán của mô hình trên ảnh ngẫu nhiên từ tập kiểm tra cùng với đồ thị
def visualize_random_images_with_graphs(model, test_loader, class_names, num_images_per_row=7, num_rows=4):
    was_training = model.training
    model.eval()
    total_images = len(test_loader.dataset)

    # Kiểm tra xem có đủ số ảnh để hiển thị không
    if num_images_per_row * num_rows > total_images:
        raise ValueError("Số lượng ảnh cần hiển thị lớn hơn số lượng ảnh trong tập kiểm tra.")

    # Chọn ngẫu nhiên các ảnh
    selected_indices = random.sample(range(total_images), num_images_per_row * num_rows)

    plt.figure(figsize=(30,200))
    plt.subplots_adjust(hspace=0.5)

    with torch.no_grad():
        for i, idx in enumerate(selected_indices):
            input_img, true_label = test_loader.dataset[idx]
            input_img = input_img.unsqueeze(0).to(device)

            # Dự đoán
            output = model(input_img)
            _, pred_label = torch.max(output, 1)

            # Hiển thị ảnh và dự đoán
            plt.subplot(28, num_images_per_row * 2, i * 2 + 1)
            plt.imshow(transforms.ToPILImage()(input_img.cpu().squeeze()))
            plt.title(f'Pred: {class_names[pred_label.item()]} \nTrue: {class_names[true_label]}',color = "green")
            plt.axis('off')

            # Hiển thị đồ thị
            plt.subplot(28, num_images_per_row * 2, i * 2 + 2)
            data = output.squeeze().cpu().numpy()
            plt.bar(class_names, data)
            plt.xticks(rotation='vertical')
            plt.title("Score")

    plt.show()
    model.train(mode=was_training)

# Hiển thị dự đoán của mô hình trên ảnh ngẫu nhiên cùng với đồ thị (28 ảnh trên 4 dòng)
visualize_random_images_with_graphs(model, test_loader, class_names, num_images_per_row=5, num_rows=5)


'''# Hàm để hiển thị một batch của dữ liệu kiểm tra cùng với dự đoán của mô hình
def visualize_random_images(model, test_loader, class_names, num_images_per_row=8):
    was_training = model.training
    model.eval()
    total_images = len(test_loader.dataset)

    # Chọn ngẫu nhiên các ảnh
    selected_indices = random.sample(range(total_images), num_images_per_row)

    # Tính số dòng cần tạo dựa trên số ảnh trên mỗi dòng
    num_rows = (num_images_per_row + 1) // 2

    plt.figure(figsize=(15, 3 * num_rows))

    with torch.no_grad():
        for i, idx in enumerate(selected_indices):
            input_img, true_label = test_loader.dataset[idx]
            input_img = input_img.unsqueeze(0).to(device)

            # Dự đoán
            output = model(input_img)
            _, pred_label = torch.max(output, 1)

            # Hiển thị ảnh và dự đoán
            plt.subplot(num_rows, num_images_per_row, i + 1)
            plt.imshow(transforms.ToPILImage()(input_img.cpu().squeeze()))
            plt.title(f'Pred: {class_names[pred_label.item()]} \nTrue: {class_names[true_label]}')
            plt.axis('off')

    plt.show()
    model.train(mode=was_training)

# Hiển thị một số ảnh kiểm tra cùng với dự đoán của mô hình
plt.figure(figsize=(15, 8))
visualize_model(model, test_loader, class_names, num_images=16)
plt.show()

# Hiển thị dự đoán của mô hình trên ảnh ngẫu nhiên
visualize_random_images(model, test_loader, class_names, num_images_per_row=16)'''


Output hidden; open in https://colab.research.google.com to view.